# 0.11.1 ESC Policy-Focused DCE Workflow

**Purpose**: Redesigned discrete choice experiment (DCE) workflow focused on ESC policy evaluation

**Policy Question**: How can we convert non-beneficiaries to ESC beneficiaries in private schools through voucher increases?

**Key Design Decisions**:
- **Primary lever**: ESC voucher amount (current: 9,000 PHP)
- **Success metric**: % of non-beneficiaries switching from public to ESC private schools
- **Model scope**: 5km radius around each student (computational efficiency)
- **Feature focus**: Out-of-pocket cost after ESC subsidy, not beneficiary status interactions

**Workflow**:
1. **Task 3.1**: Build DCE dataset with ESC policy features
2. **Task 3.2**: Estimate choice model on current ESC beneficiaries
3. **Task 3.3**: Validate model predictions
4. **Task 3.4**: Simulate ESC policy scenarios
5. **Prototype**: City of Antipolo end-to-end demonstration

**Expected Outputs**:
- Coefficient estimates showing cost sensitivity
- Conversion predictions under different voucher amounts (12k, 15k PHP)
- Identification of high-demand schools for ESC expansion

---
## Setup

In [1]:
import os
import sys
import json
from importlib import reload
import numpy as np
import pandas as pd
import geopandas as gpd
from pathlib import Path
from scipy.spatial import cKDTree
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Add project root to Python path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Import config
from config import setup_notebook, get_path
setup_notebook()

# Set display options
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", None)

print("✓ Setup complete")

✓ Project root: /workspace/project_paaral
✓ Working directory: /workspace/project_paaral
✓ Python path updated
✓ Setup complete


In [2]:
print("="*80)
print("ESC POLICY-FOCUSED DCE WORKFLOW")
print("="*80)
print("\nPolicy Question:")
print("  How can we convert non-beneficiaries to ESC beneficiaries through voucher increases?")
print("\nKey Parameters:")
print("  - Current ESC voucher: 9,000 PHP")
print("  - Choice set radius: 5 km")
print("  - Target province: Rizal (PH04058)")
print("  - Model features: 10 (focused on cost, ESC participation, access)")

ESC POLICY-FOCUSED DCE WORKFLOW

Policy Question:
  How can we convert non-beneficiaries to ESC beneficiaries through voucher increases?

Key Parameters:
  - Current ESC voucher: 9,000 PHP
  - Choice set radius: 5 km
  - Target province: Rizal (PH04058)
  - Model features: 10 (focused on cost, ESC participation, access)


---
## Task 3.1: Build DCE Dataset with ESC Policy Features

**Objective**: Create choice dataset with features that capture ESC policy levers

**Key Features**:
1. **`out_of_pocket_cost_thousands`** - (Tuition - 9000) / 1000 [PRIMARY LEVER]
2. **`is_fully_covered`** - Binary: tuition ≤ 9000 PHP
3. **`is_esc_participating`** - School accepts ESC vouchers
4. **`school_size_*`** - Medium/Large/Very Large dummies
5. **`capacity_utilization`** - Enrollment/seats ratio
6. **`log_distance_km`** - Log(road network distance)
7. **`same_sector`** - Origin and destination same sector
8. **`esc_x_fully_covered`** - Interaction: ESC school that's fully covered

**Data Sources**:
- Unified Grade 7 flow data (student choices)
- School coordinates (Modules 2, 3)
- Tuition data (Module 6) + nearest neighbor imputation
- Distance matrix (Module 12 - Rizal provincial network)
- Enrollment/seats (Modules 1, 4)

### Step 1.1: Load Student Flow Data

In [3]:
print("\n" + "="*80)
print("STEP 1.1: LOADING STUDENT FLOW DATA")
print("="*80)

# Load unified Grade 7 dataset (aggregated by school pair)
# TODO: Replace with actual aggregated unified Grade 7 data path
flow_data_path = 'output/unified_gr7_flows_aggregated.csv'  # Placeholder

# For now, we'll use a placeholder structure
# Expected columns: school_id_origin, school_id_destination, 
#                   beneficiary_count, non_beneficiary_count, total_student_count

print("\nNote: This step requires aggregated unified Grade 7 flow data")
print("      Expected format: origin-destination pairs with beneficiary counts")
print("\nPlaceholder: We'll create synthetic data structure for demonstration")

# We'll load actual student-level data in the prototype section


STEP 1.1: LOADING STUDENT FLOW DATA

Note: This step requires aggregated unified Grade 7 flow data
      Expected format: origin-destination pairs with beneficiary counts

Placeholder: We'll create synthetic data structure for demonstration


### Step 1.2: Load School Data

In [5]:
print("\n" + "="*80)
print("STEP 1.2: LOADING SCHOOL DATA")
print("="*80)

# Load school node tables (from Module 11)
print("\n[1/4] Loading school node tables...")
node_tables_path = 'output/all_nodes_valid.gpkg'

if os.path.exists(node_tables_path):
    schools_gdf = gpd.read_file(node_tables_path)
    print(f"✓ Loaded {len(schools_gdf):,} schools")
    
    # Filter to Rizal province
    rizal_schools = schools_gdf[schools_gdf['adm2_pcode'] == 'PH04058'].copy()
    print(f"✓ Filtered to Rizal: {len(rizal_schools):,} schools")
    
    # Display sample
    print("\nSample school data:")
    print(rizal_schools[['school_id', 'sector', 'total_enrollment', 'municipality']].head())
else:
    print(f"⚠ Node table file not found: {node_tables_path}")
    print("  Run notebook 0.7 to generate node tables first")
    rizal_schools = None


STEP 1.2: LOADING SCHOOL DATA

[1/4] Loading school node tables...
✓ Loaded 54,204 schools
✓ Filtered to Rizal: 778 schools

Sample school data:
      school_id  sector  total_enrollment municipality
10171    108460  public             284.0        Tanay
10174    108463  public              84.0        Tanay
11127    109307  public            3751.0       Angono
11128    109310  public            1202.0       Angono
11129    109311  public            3274.0       Angono


### Step 1.3: Load Tuition Data and Impute Missing Values

In [6]:
# from modules import subsidy_tuition_processor
# # Reload the module
# reload(subsidy_tuition_processor)

# from modules import subsidy_tuition_processor as stp

# # Initialize processor
# processor_stp = stp.SubsidyTuitionProcessor(verbose=False)

# # Process both datasets
# esc_data, shsvp_data = processor_stp.process()

# # Export processed data
# processor_stp.export_processed()

In [7]:
print("\n" + "="*80)
print("STEP 1.3: LOADING TUITION DATA")
print("="*80)

# Load tuition data from Module 6 (ESC and SHSVP tuition)
print("\n[1/3] Loading ESC tuition data...")
esc_tuition_path = 'output/esc_tuition_long.csv'  # From Module 6

if os.path.exists(esc_tuition_path):
    esc_tuition = pd.read_csv(esc_tuition_path)
    print(f"✓ Loaded {len(esc_tuition):,} ESC tuition records")
    
    # Convert school_id to string for compatibility with node tables
    esc_tuition['school_id'] = esc_tuition['school_id'].astype(str)
    
    # Filter to Grade 7 (JHS level)
    esc_gr7 = esc_tuition[esc_tuition['grade_level'] == 'G7'].copy()
    print(f"✓ Filtered to Grade 7: {len(esc_gr7):,} records")
    
    # Pivot fee_type to columns
    print("✓ Pivoting fee types to columns...")
    esc_gr7_pivot = esc_gr7.pivot_table(
        index='school_id',
        columns='fee_type',
        values='amount',
        aggfunc='mean',  # Average if multiple entries
        fill_value=0
    ).reset_index()
    
    # Calculate total cost
    school_tuition = esc_gr7_pivot.copy()
    school_tuition['total_cost'] = (
        school_tuition.get('Tuition', 0) + 
        school_tuition.get('Other', 0) + 
        school_tuition.get('Miscellaneous', 0)
    )
    
    # Rename columns for clarity
    school_tuition = school_tuition.rename(columns={
        'Tuition': 'tuition',
        'Other': 'other_fees',
        'Miscellaneous': 'miscellaneous'
    })
    
    print(f"✓ Aggregated to {len(school_tuition):,} unique schools")
    print(f"\nTuition statistics (PHP):")
    print(school_tuition['total_cost'].describe())
else:
    print(f"⚠ Tuition data not found: {esc_tuition_path}")
    print("  Using placeholder median tuition values")
    school_tuition = None


STEP 1.3: LOADING TUITION DATA

[1/3] Loading ESC tuition data...
✓ Loaded 39,600 ESC tuition records
✓ Filtered to Grade 7: 9,920 records
✓ Pivoting fee types to columns...
✓ Aggregated to 3,614 unique schools

Tuition statistics (PHP):
count      3614.000000
mean      27668.832340
std       18776.053476
min        2200.000000
25%       14500.000000
50%       22600.000000
75%       34993.750000
max      238195.000000
Name: total_cost, dtype: float64


In [8]:
print("\n[2/3] Imputing missing tuition values using nearest neighbor...")

if rizal_schools is not None and school_tuition is not None:
    # Merge tuition data with schools
    rizal_schools_with_tuition = rizal_schools.merge(
        school_tuition[['school_id', 'total_cost']],
        on='school_id',
        how='left'
    )
    
    # Check coverage
    n_with_tuition = rizal_schools_with_tuition['total_cost'].notna().sum()
    n_missing = rizal_schools_with_tuition['total_cost'].isna().sum()
    
    print(f"  Schools with tuition data: {n_with_tuition:,}")
    print(f"  Schools missing tuition: {n_missing:,}")
    
    if n_missing > 0:
        print(f"\n  Applying nearest neighbor imputation for {n_missing:,} schools...")
        
        # Strategy: For each school without tuition, find nearest school with tuition (same sector)
        # Get schools with tuition
        schools_with_tuition = rizal_schools_with_tuition[rizal_schools_with_tuition['total_cost'].notna()].copy()
        schools_missing_tuition = rizal_schools_with_tuition[rizal_schools_with_tuition['total_cost'].isna()].copy()
        
        # For each sector, build KDTree
        for sector in ['public', 'private']:
            sector_with = schools_with_tuition[schools_with_tuition['sector'] == sector]
            sector_missing = schools_missing_tuition[schools_missing_tuition['sector'] == sector]
            
            if len(sector_with) > 0 and len(sector_missing) > 0:
                # Build KDTree from schools with tuition
                coords_with = np.column_stack([
                    sector_with.geometry.x.values,
                    sector_with.geometry.y.values
                ])
                
                coords_missing = np.column_stack([
                    sector_missing.geometry.x.values,
                    sector_missing.geometry.y.values
                ])
                
                tree = cKDTree(coords_with)
                distances, indices = tree.query(coords_missing, k=1)
                
                # Assign nearest neighbor tuition
                nearest_tuition = sector_with.iloc[indices]['total_cost'].values
                
                # Update in original dataframe
                rizal_schools_with_tuition.loc[
                    rizal_schools_with_tuition['school_id'].isin(sector_missing['school_id']),
                    'total_cost'
                ] = nearest_tuition
                
                print(f"    ✓ Imputed {len(sector_missing):,} {sector} schools (avg distance: {distances.mean():.2f}°)")
        
        print(f"\n  ✓ Imputation complete")
    
    # Rename column
    rizal_schools_with_tuition.rename(columns={'total_cost': 'school_tuition'}, inplace=True)
    
    print(f"\n✓ All schools now have tuition estimates")
else:
    print("  ⚠ Skipping imputation (missing input data)")
    rizal_schools_with_tuition = rizal_schools


[2/3] Imputing missing tuition values using nearest neighbor...
  Schools with tuition data: 111
  Schools missing tuition: 667

  Applying nearest neighbor imputation for 667 schools...
    ✓ Imputed 308 private schools (avg distance: 0.01°)

  ✓ Imputation complete

✓ All schools now have tuition estimates


In [9]:
print("\n[3/3] Setting public school tuition to 0...")

if rizal_schools_with_tuition is not None:
    # Public schools have no tuition
    rizal_schools_with_tuition.loc[
        rizal_schools_with_tuition['sector'] == 'public',
        'school_tuition'
    ] = 0.0
    
    print("✓ Public school tuition set to 0")
    
    # Summary statistics by sector
    print("\nTuition by sector:")
    print(rizal_schools_with_tuition.groupby('sector')['school_tuition'].describe())


[3/3] Setting public school tuition to 0...
✓ Public school tuition set to 0

Tuition by sector:
         count          mean           std      min       25%      50%  \
sector                                                                   
private  419.0  37602.304821  17557.782389  9400.26  26314.28  33983.0   
public   359.0      0.000000      0.000000     0.00      0.00      0.0   

             75%       max  
sector                      
private  43028.0  146889.0  
public       0.0       0.0  


### Step 1.4: Load Distance Matrix

In [11]:
print("\n" + "="*80)
print("STEP 1.4: LOADING DISTANCE MATRIX")
print("="*80)

# Load Rizal provincial distance matrix (from Module 12)
distance_matrix_path = 'output/PH04058_rizal_distance_matrix.csv'

if os.path.exists(distance_matrix_path):
    print("\n[1/2] Loading distance matrix...")
    distance_matrix = pd.read_csv(distance_matrix_path, index_col=0)
    
    print(f"✓ Loaded distance matrix: {distance_matrix.shape[0]} × {distance_matrix.shape[1]}")
    print(f"  Total origin-destination pairs: {distance_matrix.notna().sum().sum():,}")
    
    # Convert to dictionary for fast lookup
    print("\n[2/2] Converting to dictionary for fast lookup...")
    distance_dict = {}
    
    for origin_id in distance_matrix.index:
        for dest_id in distance_matrix.columns:
            dist_m = distance_matrix.loc[origin_id, dest_id]
            if pd.notna(dist_m):
                distance_dict[(str(origin_id), str(dest_id))] = dist_m / 1000.0  # Convert to km
    
    print(f"✓ Created distance dictionary with {len(distance_dict):,} pairs")
    
    # Sample distances
    sample_distances = list(distance_dict.values())[:1000]
    print(f"\nDistance statistics (km):")
    print(f"  Mean: {np.mean(sample_distances):.2f} km")
    print(f"  Median: {np.median(sample_distances):.2f} km")
    print(f"  Max: {np.max(sample_distances):.2f} km")
else:
    print(f"⚠ Distance matrix not found: {distance_matrix_path}")
    print("  Run notebook 0.9.1 to generate Rizal provincial network first")
    distance_dict = None


STEP 1.4: LOADING DISTANCE MATRIX

[1/2] Loading distance matrix...
✓ Loaded distance matrix: 778 × 778
  Total origin-destination pairs: 43,533

[2/2] Converting to dictionary for fast lookup...
✓ Created distance dictionary with 43,533 pairs

Distance statistics (km):
  Mean: 18.85 km
  Median: 20.40 km
  Max: 29.96 km


### Step 1.5: Identify ESC Participating Schools

In [12]:
print("\n" + "="*80)
print("STEP 1.5: IDENTIFYING ESC PARTICIPATING SCHOOLS")
print("="*80)

# ESC schools are identified from tuition data (Module 6)
# Schools in ESC tuition dataset are ESC participants

if rizal_schools_with_tuition is not None and school_tuition is not None:
    # Mark schools as ESC participating
    esc_school_ids = set(school_tuition['school_id'].unique())
    
    rizal_schools_with_tuition['is_esc_participating'] = (
        rizal_schools_with_tuition['school_id'].isin(esc_school_ids)
    ).astype(int)
    
    # Public schools don't participate in ESC (by definition)
    rizal_schools_with_tuition.loc[
        rizal_schools_with_tuition['sector'] == 'public',
        'is_esc_participating'
    ] = 0
    
    n_esc = rizal_schools_with_tuition['is_esc_participating'].sum()
    n_private = (rizal_schools_with_tuition['sector'] == 'private').sum()
    
    print(f"✓ Identified {n_esc:,} ESC participating schools")
    print(f"  Out of {n_private:,} private schools ({100*n_esc/n_private:.1f}%)")
    
    # Summary
    print("\nSchool distribution:")
    print(rizal_schools_with_tuition.groupby(['sector', 'is_esc_participating']).size())
else:
    print("⚠ Skipping ESC identification (missing input data)")


STEP 1.5: IDENTIFYING ESC PARTICIPATING SCHOOLS
✓ Identified 111 ESC participating schools
  Out of 419 private schools (26.5%)

School distribution:
sector   is_esc_participating
private  0                       308
         1                       111
public   0                       359
dtype: int64


### Step 1.6: Calculate ESC Policy Features

In [13]:
print("\n" + "="*80)
print("STEP 1.6: CALCULATING ESC POLICY FEATURES")
print("="*80)

# ESC voucher amount
ESC_VOUCHER_AMOUNT = 9000  # PHP

if rizal_schools_with_tuition is not None:
    print(f"\n[1/4] Calculating out-of-pocket cost (tuition - {ESC_VOUCHER_AMOUNT:,} PHP)...")
    
    # Out-of-pocket cost after ESC subsidy
    rizal_schools_with_tuition['out_of_pocket_cost'] = (
        rizal_schools_with_tuition['school_tuition'] - ESC_VOUCHER_AMOUNT
    )
    
    # Convert to thousands for model estimation
    rizal_schools_with_tuition['out_of_pocket_cost_thousands'] = (
        rizal_schools_with_tuition['out_of_pocket_cost'] / 1000.0
    )
    
    # For public schools, set to 0 (no cost)
    rizal_schools_with_tuition.loc[
        rizal_schools_with_tuition['sector'] == 'public',
        'out_of_pocket_cost_thousands'
    ] = 0.0
    
    print(f"✓ Calculated out-of-pocket cost")
    print(f"\nOut-of-pocket cost distribution (thousands PHP):")
    print(rizal_schools_with_tuition['out_of_pocket_cost_thousands'].describe())
    
    print(f"\n[2/4] Identifying fully covered schools (tuition ≤ {ESC_VOUCHER_AMOUNT:,} PHP)...")
    
    # Schools fully covered by ESC voucher
    rizal_schools_with_tuition['is_fully_covered'] = (
        rizal_schools_with_tuition['school_tuition'] <= ESC_VOUCHER_AMOUNT
    ).astype(int)
    
    n_fully_covered = rizal_schools_with_tuition['is_fully_covered'].sum()
    n_total = len(rizal_schools_with_tuition)
    
    print(f"✓ Identified {n_fully_covered:,} fully covered schools ({100*n_fully_covered/n_total:.1f}%)")
    
    print(f"\n[3/4] Creating interaction term (ESC × Fully Covered)...")
    
    # Interaction: ESC school that's fully covered ("sweet spot" schools)
    rizal_schools_with_tuition['esc_x_fully_covered'] = (
        rizal_schools_with_tuition['is_esc_participating'] * 
        rizal_schools_with_tuition['is_fully_covered']
    )
    
    n_sweet_spot = rizal_schools_with_tuition['esc_x_fully_covered'].sum()
    print(f"✓ Identified {n_sweet_spot:,} 'sweet spot' schools (ESC + fully covered)")
    
    print(f"\n[4/4] Calculating school size categories...")
    
    # Define size thresholds based on total enrollment
    # Small: < 300, Medium: 300-600, Large: 600-1000, Very Large: > 1000
    rizal_schools_with_tuition['school_size_medium'] = (
        (rizal_schools_with_tuition['total_enrollment'] >= 300) &
        (rizal_schools_with_tuition['total_enrollment'] < 600)
    ).astype(int)
    
    rizal_schools_with_tuition['school_size_large'] = (
        (rizal_schools_with_tuition['total_enrollment'] >= 600) &
        (rizal_schools_with_tuition['total_enrollment'] < 1000)
    ).astype(int)
    
    rizal_schools_with_tuition['school_size_very_large'] = (
        rizal_schools_with_tuition['total_enrollment'] >= 1000
    ).astype(int)
    
    print(f"✓ School size distribution:")
    size_dist = pd.DataFrame({
        'Small': ((rizal_schools_with_tuition['school_size_medium'] == 0) &
                  (rizal_schools_with_tuition['school_size_large'] == 0) &
                  (rizal_schools_with_tuition['school_size_very_large'] == 0)).sum(),
        'Medium': rizal_schools_with_tuition['school_size_medium'].sum(),
        'Large': rizal_schools_with_tuition['school_size_large'].sum(),
        'Very Large': rizal_schools_with_tuition['school_size_very_large'].sum()
    }, index=['Count'])
    print(size_dist.T)
    
    print(f"\n✓ All ESC policy features calculated")
else:
    print("⚠ Skipping feature calculation (missing input data)")


STEP 1.6: CALCULATING ESC POLICY FEATURES

[1/4] Calculating out-of-pocket cost (tuition - 9,000 PHP)...
✓ Calculated out-of-pocket cost

Out-of-pocket cost distribution (thousands PHP):
count    778.000000
mean      15.404069
std       19.220047
min        0.000000
25%        0.000000
50%       11.437610
75%       27.790000
max      137.889000
Name: out_of_pocket_cost_thousands, dtype: float64

[2/4] Identifying fully covered schools (tuition ≤ 9,000 PHP)...
✓ Identified 359 fully covered schools (46.1%)

[3/4] Creating interaction term (ESC × Fully Covered)...
✓ Identified 0 'sweet spot' schools (ESC + fully covered)

[4/4] Calculating school size categories...
✓ School size distribution:
            Count
Small         380
Medium         94
Large          77
Very Large    227

✓ All ESC policy features calculated


### Step 1.7: Build Choice Set for Each Student (5km Radius)

In [14]:
print("\n" + "="*80)
print("STEP 1.7: BUILDING CHOICE SETS (5KM RADIUS)")
print("="*80)

print("\nNote: This step creates learner × alternative choice sets")
print("      Computational scope: Students only consider schools within 5km")
print("\n⚠ This is a computationally intensive step")
print("  Full Rizal: 48,585 students × ~50 schools/student = ~2.4M choice alternatives")
print("  We'll implement this in the City of Antipolo prototype section")
print("\n✓ Deferred to prototype section below")


STEP 1.7: BUILDING CHOICE SETS (5KM RADIUS)

Note: This step creates learner × alternative choice sets
      Computational scope: Students only consider schools within 5km

⚠ This is a computationally intensive step
  Full Rizal: 48,585 students × ~50 schools/student = ~2.4M choice alternatives
  We'll implement this in the City of Antipolo prototype section

✓ Deferred to prototype section below


---
## Task 3.2: Estimate Choice Model on Current ESC Beneficiaries

**Objective**: Estimate multinomial logit model using current beneficiaries' revealed preferences

**Model Specification**:
```
Utility(school j, student i) = 
    β₁ × out_of_pocket_cost_thousands[j]
  + β₂ × is_fully_covered[j]
  + β₃ × is_esc_participating[j]
  + β₄ × school_size_medium[j]
  + β₅ × school_size_large[j]
  + β₆ × school_size_very_large[j]
  + β₇ × capacity_utilization[j]
  + β₈ × log_distance_km[i,j]
  + β₉ × same_sector[i,j]
  + β₁₀ × esc_x_fully_covered[j]
```

**Expected Coefficient Signs**:
- β₁ < 0 (negative) - Higher out-of-pocket cost reduces utility
- β₂ > 0 (positive) - Fully covered schools more attractive
- β₃ > 0 (positive) - ESC participating schools preferred
- β₈ < 0 (negative) - Distance penalty
- β₉ > 0 (positive) - Same sector preference (inertia)
- β₁₀ > 0 (positive) - ESC + fully covered especially attractive

In [15]:
print("\n" + "="*80)
print("TASK 3.2: ESTIMATE CHOICE MODEL")
print("="*80)

print("\nEstimation approach:")
print("  1. Filter DCE dataset to current ESC beneficiaries only")
print("  2. Estimate multinomial logit using statsmodels.discrete.discrete_model")
print("  3. Extract coefficients and standard errors")
print("  4. Validate model fit (McFadden R², log-likelihood ratio test)")
print("\n✓ Implementation deferred to prototype section")


TASK 3.2: ESTIMATE CHOICE MODEL

Estimation approach:
  1. Filter DCE dataset to current ESC beneficiaries only
  2. Estimate multinomial logit using statsmodels.discrete.discrete_model
  3. Extract coefficients and standard errors
  4. Validate model fit (McFadden R², log-likelihood ratio test)

✓ Implementation deferred to prototype section


In [16]:
# Placeholder for model estimation code
# Will be implemented in prototype

# from statsmodels.discrete.discrete_model import MNLogit

# # Filter to beneficiaries
# dce_beneficiaries = dce_data[dce_data['is_current_beneficiary'] == 1].copy()

# # Prepare feature matrix
# X = dce_beneficiaries[[
#     'out_of_pocket_cost_thousands',
#     'is_fully_covered',
#     'is_esc_participating',
#     'school_size_medium',
#     'school_size_large',
#     'school_size_very_large',
#     'capacity_utilization',
#     'log_distance_km',
#     'same_sector',
#     'esc_x_fully_covered'
# ]]

# # Choice variable (which school was chosen)
# y = dce_beneficiaries['chosen']

# # Estimate model
# model = MNLogit(y, X)
# results = model.fit()

# # Display results
# print(results.summary())

---
## Task 3.3: Validate Model Predictions

**Validation Approach**:
1. **In-sample prediction accuracy**: Compare predicted vs actual choices for beneficiaries
2. **Choice probability distribution**: Verify probabilities sum to 1 for each student
3. **Coefficient reasonableness**: Check signs and magnitudes against expectations
4. **Market share validation**: Compare predicted enrollment shares to actual data

In [17]:
print("\n" + "="*80)
print("TASK 3.3: MODEL VALIDATION")
print("="*80)

print("\nValidation metrics:")
print("  1. Prediction accuracy: % of students whose actual choice has highest predicted probability")
print("  2. Coefficient signs: All coefficients match theoretical expectations")
print("  3. McFadden R²: Measure of model fit (>0.2 is good for choice models)")
print("  4. Market shares: Predicted public vs private enrollment matches actual")
print("\n✓ Implementation deferred to prototype section")


TASK 3.3: MODEL VALIDATION

Validation metrics:
  1. Prediction accuracy: % of students whose actual choice has highest predicted probability
  2. Coefficient signs: All coefficients match theoretical expectations
  3. McFadden R²: Measure of model fit (>0.2 is good for choice models)
  4. Market shares: Predicted public vs private enrollment matches actual

✓ Implementation deferred to prototype section


---
## Task 3.4: Simulate ESC Policy Scenarios

**Policy Scenarios**:

### **Baseline: Current ESC Voucher (9,000 PHP)**
- No changes to voucher amount or school participation
- Expected: ~5,674 students choose ESC schools (current beneficiaries)

### **Scenario 1: Increase ESC Voucher to 12,000 PHP (+33%)**
- Recalculate: `out_of_pocket_cost = tuition - 12,000`
- More schools become `is_fully_covered = 1`
- **Measure**: How many non-beneficiaries switch to ESC schools?

### **Scenario 2: Increase ESC Voucher to 15,000 PHP (+67%)**
- Recalculate: `out_of_pocket_cost = tuition - 15,000`
- Most low-to-mid tier private schools fully covered
- **Measure**: Additional conversions beyond 12k scenario

### **Scenario 3: Expand ESC to High-Demand Non-ESC Schools**
- Identify top 10 non-ESC schools with highest predicted demand
- Set `is_esc_participating = 1` for these schools
- **Measure**: How many students switch to newly available ESC schools?

### **Scenario 4: Combined (15k Voucher + Expanded Participation)**
- Both interventions together
- **Measure**: Total public → ESC conversions

In [18]:
print("\n" + "="*80)
print("TASK 3.4: SIMULATE POLICY SCENARIOS")
print("="*80)

print("\nSimulation approach:")
print("  1. Use estimated coefficients from Task 3.2")
print("  2. For each scenario, recalculate affected features")
print("  3. Compute choice probabilities for ALL students (not just beneficiaries)")
print("  4. Sample choices probabilistically")
print("  5. Compare baseline vs scenario enrollment patterns")
print("\nSuccess metric:")
print("  Number of non-beneficiaries who switch from public to ESC private schools")
print("\n✓ Implementation deferred to prototype section")


TASK 3.4: SIMULATE POLICY SCENARIOS

Simulation approach:
  1. Use estimated coefficients from Task 3.2
  2. For each scenario, recalculate affected features
  3. Compute choice probabilities for ALL students (not just beneficiaries)
  4. Sample choices probabilistically
  5. Compare baseline vs scenario enrollment patterns

Success metric:
  Number of non-beneficiaries who switch from public to ESC private schools

✓ Implementation deferred to prototype section


---
## PROTOTYPE: City of Antipolo End-to-End Demonstration

**Purpose**: Complete workflow demonstration on a manageable subset

**City of Antipolo Context**:
- Most populous city in Rizal (~887k population)
- Mix of urban and semi-urban areas
- Significant private school presence
- Ideal test case for ESC policy analysis

**Prototype Scope**:
- All Grade 7 students in Antipolo schools
- 5km choice radius
- Complete Tasks 3.1 → 3.2 → 3.3 → 3.4
- Full policy scenario comparison

### Prototype Step 1: Filter to City of Antipolo

In [20]:
print("\n" + "="*80)
print("PROTOTYPE: CITY OF ANTIPOLO")
print("="*80)

if rizal_schools_with_tuition is not None:
    # Filter schools to Antipolo
    antipolo_schools = rizal_schools_with_tuition[
        rizal_schools_with_tuition['municipality'].str.contains('Antipolo', case=False, na=False)
    ].copy()
    
    print(f"\n✓ Filtered to City of Antipolo: {len(antipolo_schools):,} schools")
    print(f"\nSchool distribution:")
    print(antipolo_schools['sector'].value_counts())
    
    print(f"\nESC participation:")
    print(antipolo_schools.groupby(['sector', 'is_esc_participating']).size())
    
    # Display sample
    print(f"\nSample Antipolo schools:")
    display_cols = ['school_id', 'sector', 'total_enrollment', 'school_tuition', 
                    'is_esc_participating', 'is_fully_covered', 'out_of_pocket_cost_thousands']
    print(antipolo_schools[display_cols].head(10))
else:
    print("⚠ Cannot filter to Antipolo (missing school data)")
    antipolo_schools = None


PROTOTYPE: CITY OF ANTIPOLO

✓ Filtered to City of Antipolo: 182 schools

School distribution:
sector
private    115
public      67
Name: count, dtype: int64

ESC participation:
sector   is_esc_participating
private  0                       78
         1                       37
public   0                       67
dtype: int64

Sample Antipolo schools:
    school_id  sector  total_enrollment  school_tuition  is_esc_participating  \
16     109362  public            2085.0             0.0                     0   
216    165515  public              41.0             0.0                     0   
291    109342  public            8983.0             0.0                     0   
292    301418  public               0.0             0.0                     0   
293    342175  public            2966.0             0.0                     0   
294    109327  public            7472.0             0.0                     0   
295    301457  public            8410.0             0.0                     0

### Prototype Step 2: Load Antipolo Student Flow Data

In [22]:
print("\n[Prototype Step 2] Loading student flow data for Antipolo...")

# TODO: Load actual unified Grade 7 student-level data
# Filter to students whose origin school is in Antipolo

print("\n⚠ Note: This requires student-level unified Grade 7 dataset")
print("  Expected columns: learner_id, school_id_origin, school_id_destination, is_beneficiary")
print("\n  For this prototype, we'll create a synthetic dataset to demonstrate the workflow")


[Prototype Step 2] Loading student flow data for Antipolo...

⚠ Note: This requires student-level unified Grade 7 dataset
  Expected columns: learner_id, school_id_origin, school_id_destination, is_beneficiary

  For this prototype, we'll create a synthetic dataset to demonstrate the workflow


In [ ]:
# NOTE: This cell is no longer needed - we now load actual unified Grade 7 data in the cell above
# Kept as placeholder to maintain cell numbering

### Prototype Step 3: Build Choice Sets (5km Radius)

In [ ]:
print("\n[Prototype Step 2] Loading student flow data for Antipolo...")

# Load actual unified Grade 7 student-level data
unified_gr7_path = 'output/unified_gr7_flows.csv'

if os.path.exists(unified_gr7_path) and antipolo_schools is not None:
    print(f"\n✓ Loading unified Grade 7 data from {unified_gr7_path}...")
    unified_gr7 = pd.read_csv(unified_gr7_path)
    
    # Convert school_id columns to string for consistency
    unified_gr7['school_id_origin'] = unified_gr7['school_id_origin'].astype(str)
    unified_gr7['school_id_destination'] = unified_gr7['school_id_destination'].astype(str)
    
    print(f"✓ Loaded {len(unified_gr7):,} Grade 7 student flow records")
    
    # Filter to students whose origin school is in Antipolo
    antipolo_school_ids = set(antipolo_schools['school_id'].unique())
    students_antipolo = unified_gr7[
        unified_gr7['school_id_origin'].isin(antipolo_school_ids)
    ].copy()
    
    print(f"✓ Filtered to Antipolo origin schools: {len(students_antipolo):,} students")
    print(f"  Beneficiaries: {students_antipolo['is_beneficiary'].sum():,} ({100*students_antipolo['is_beneficiary'].mean():.1f}%)")
    
    # Add origin school sector
    students_antipolo = students_antipolo.merge(
        antipolo_schools[['school_id', 'sector']].rename(columns={'sector': 'sector_origin'}),
        left_on='school_id_origin',
        right_on='school_id',
        how='left'
    ).drop(columns=['school_id'])
    
    print(f"\n✓ Origin sector distribution:")
    print(students_antipolo['sector_origin'].value_counts())
    
else:
    if not os.path.exists(unified_gr7_path):
        print(f"\n⚠ Unified Grade 7 data not found: {unified_gr7_path}")
        print("  Run the unified_gr7_flow_builder module to generate this file")
    else:
        print("\n⚠ Cannot load student data (missing Antipolo school data)")
    students_antipolo = None

### Prototype Step 3: Build Choice Sets (5km Radius)

In [ ]:
print("\n[Prototype Step 3] Building choice sets (5km radius)...")

if students_antipolo is not None and antipolo_schools is not None and distance_dict is not None:
    print("\nCreating learner × alternative choice sets...")
    
    choice_sets = []
    
    for idx, student in students_antipolo.iterrows():
        lrn = student['lrn']
        origin_id = str(student['school_id_origin'])
        dest_id = str(student['school_id_destination'])  # Actual observed choice
        
        # Find all schools within 5km of origin (these are the alternatives)
        for alt_school in antipolo_schools.itertuples():
            alt_id = str(alt_school.school_id)
            
            # Lookup distance from origin to this alternative
            distance_km = distance_dict.get((origin_id, alt_id), None)
            
            if distance_km is not None and distance_km <= 5.0:
                # Add to choice set
                choice_sets.append({
                    'lrn': lrn,
                    'school_id_origin': origin_id,
                    'school_id_alternative': alt_id,
                    'school_id_destination': dest_id,  # Actual choice
                    'distance_km': distance_km,
                    'log_distance_km': np.log(max(distance_km, 0.1)),  # Avoid log(0)
                    'is_current_beneficiary': student['is_beneficiary'],
                    'sector_origin': student['sector_origin']
                })
    
    dce_antipolo = pd.DataFrame(choice_sets)
    
    print(f"\n✓ Created choice set: {len(dce_antipolo):,} learner-alternative pairs")
    print(f"  Avg alternatives per student: {len(dce_antipolo) / len(students_antipolo):.1f}")
    
    # Merge school features for alternatives
    school_features = [
        'school_id', 'sector', 'school_tuition', 'out_of_pocket_cost_thousands',
        'is_fully_covered', 'is_esc_participating', 'esc_x_fully_covered',
        'school_size_medium', 'school_size_large', 'school_size_very_large',
        'capacity_utilization'
    ]
    
    dce_antipolo = dce_antipolo.merge(
        antipolo_schools[school_features].rename(columns={'sector': 'sector_destination'}),
        left_on='school_id_alternative',
        right_on='school_id',
        how='left'
    ).drop(columns=['school_id'])
    
    # Calculate same_sector
    dce_antipolo['same_sector'] = (
        dce_antipolo['sector_origin'] == dce_antipolo['sector_destination']
    ).astype(int)
    
    # Mark the observed choice (chosen = 1 if alternative == actual destination)
    dce_antipolo['chosen'] = (
        dce_antipolo['school_id_alternative'] == dce_antipolo['school_id_destination']
    ).astype(int)
    
    print(f"\n✓ Merged school features")
    print(f"✓ Marked observed choices: {dce_antipolo['chosen'].sum():,} students with valid choices")
    
    # Choice distribution
    chosen_schools = dce_antipolo[dce_antipolo['chosen'] == 1]
    print(f"\nChosen school distribution:")
    print(f"  Public: {(chosen_schools['sector_destination'] == 'public').sum():,}")
    print(f"  Private (ESC): {(chosen_schools['is_esc_participating'] == 1).sum():,}")
    print(f"  Private (non-ESC): {((chosen_schools['sector_destination'] == 'private') & (chosen_schools['is_esc_participating'] == 0)).sum():,}")
    
    print(f"\nBeneficiary choices:")
    beneficiary_choices = chosen_schools[chosen_schools['is_current_beneficiary'] == 1]
    print(f"  Total beneficiaries: {len(beneficiary_choices):,}")
    print(f"  Chose ESC schools: {(beneficiary_choices['is_esc_participating'] == 1).sum():,}")
    print(f"  Chose public schools: {(beneficiary_choices['sector_destination'] == 'public').sum():,}")
    
    print(f"\nSample DCE data:")
    print(dce_antipolo[['lrn', 'school_id_alternative', 'chosen', 'distance_km', 'is_esc_participating', 'out_of_pocket_cost_thousands']].head(10))
else:
    print("\n⚠ Cannot build choice sets (missing input data)")
    if students_antipolo is None:
        print("  Missing: Unified Grade 7 student data")
        print("  Action: Ensure 'output/unified_gr7_flows.csv' exists")
    if antipolo_schools is None:
        print("  Missing: Antipolo school data")
    if distance_dict is None:
        print("  Missing: Distance matrix")
        print("  Action: Ensure 'output/PH04058_rizal_distance_matrix.csv' exists")
    dce_antipolo = None

### Prototype Step 4: Model Estimation (Ready for Implementation)

In [ ]:
print("\n[Prototype Step 4] Model estimation (ready for implementation)...")

if dce_antipolo is not None:
    print("\n✓ DCE dataset ready for model estimation")
    print(f"  Total choice observations: {dce_antipolo['chosen'].sum():,} students")
    print(f"  Beneficiary observations: {dce_antipolo[dce_antipolo['is_current_beneficiary']==1]['chosen'].sum():,} students")
    
    print("\n⚠ Model estimation requires statsmodels library")
    print("  Install with: pip install statsmodels")
    
    print("\n" + "="*70)
    print("CODE READY TO RUN (when statsmodels is installed):")
    print("="*70)
    print("""
from statsmodels.discrete.discrete_model import MNLogit

# Filter to beneficiaries only (these students revealed their ESC preferences)
dce_beneficiaries = dce_antipolo[dce_antipolo['is_current_beneficiary'] == 1].copy()

print(f"Model sample: {len(dce_beneficiaries):,} choice observations from beneficiaries")

# Prepare feature matrix
X = dce_beneficiaries[[
    'out_of_pocket_cost_thousands',
    'is_fully_covered',
    'is_esc_participating',
    'school_size_medium',
    'school_size_large',
    'school_size_very_large',
    'capacity_utilization',
    'log_distance_km',
    'same_sector',
    'esc_x_fully_covered'
]]

# Choice indicator (1 if this alternative was chosen, 0 otherwise)
y = dce_beneficiaries['chosen']

# Estimate multinomial logit model
print("\\nEstimating model...")
model = MNLogit(y, X)
results = model.fit(method='bfgs', maxiter=100)

# Display results
print("\\n" + "="*70)
print("MODEL ESTIMATION RESULTS")
print("="*70)
print(results.summary())

# Extract coefficients
coefficients = results.params
print("\\nEstimated coefficients:")
print(coefficients)
""")
    print("="*70)
else:
    print("\n⚠ Cannot estimate model (DCE dataset not ready)")
    print("  Check that unified Grade 7 data and distance matrix are available")

### Prototype Step 5: Policy Scenario Simulation

In [ ]:
print("\n[Prototype Step 5] Policy scenario simulation...")

print("\nScenario simulation workflow:")
print("  1. Baseline (9,000 PHP voucher) - current state")
print("  2. Scenario 1 (12,000 PHP voucher) - 33% increase")
print("  3. Scenario 2 (15,000 PHP voucher) - 67% increase")
print("  4. Scenario 3 (Expand ESC to top 5 high-demand schools)")
print("  5. Scenario 4 (15k voucher + expanded participation)")

print("\n⚠ Requires estimated coefficients from Step 4")
print("  Once model is estimated, use coefficients to predict choice probabilities")
print("  under different policy scenarios")

print("\n" + "="*70)
print("SIMULATION PSEUDOCODE:")
print("="*70)
print("""
# After model estimation in Step 4:
coefficients = results.params

# For each scenario:
# 1. Recalculate affected features (e.g., out_of_pocket_cost for new voucher amount)
# 2. Compute utility for each alternative using estimated coefficients
# 3. Convert utilities to choice probabilities (multinomial logit formula)
# 4. Identify students who switch from public to ESC schools
# 5. Calculate fiscal impact (voucher amount × new beneficiaries)

# Example for Scenario 1 (12k voucher):
dce_scenario1 = dce_antipolo.copy()
dce_scenario1['out_of_pocket_cost_thousands'] = (
    (dce_scenario1['school_tuition'] - 12000) / 1000
)
dce_scenario1.loc[dce_scenario1['sector_destination']=='public', 'out_of_pocket_cost_thousands'] = 0

# Compute utilities and probabilities...
# Compare with baseline to measure policy impact
""")

---
## Summary

This notebook outlines the complete ESC policy-focused DCE workflow:

### **Key Design Decisions**

1. **Feature Focus**: Out-of-pocket cost after ESC subsidy (not beneficiary interactions)
2. **Model Sample**: Estimate on current beneficiaries, simulate on all students
3. **Computational Scope**: 5km radius choice sets (manageable size)
4. **Primary Lever**: ESC voucher amount (9k → 12k → 15k PHP)
5. **Success Metric**: Non-beneficiaries switching from public to ESC private schools

### **Data Requirements**

✅ **Available**:
- School coordinates and attributes (Modules 2, 3, 11)
- Distance matrix (Module 12)
- Enrollment and capacity (Modules 1, 4)

⚠ **Need to Complete**:
- Tuition data integration (Module 6 → school-level aggregation)
- Unified Grade 7 student-level dataset
- ESC participation identification

### **Next Steps**

1. **Complete data preparation**:
   - Aggregate unified Grade 7 flows (student-level)
   - Merge tuition data from Module 6
   - Generate Rizal distance matrix if not yet available

2. **Run City of Antipolo prototype**:
   - Full end-to-end workflow on manageable subset
   - Validate model estimation and simulation
   - Refine feature specifications based on results

3. **Scale to full Rizal province**:
   - 48,585 students × ~50 schools = ~2.4M choice alternatives
   - Implement chunked processing for memory efficiency
   - Generate final policy scenario comparisons

4. **Policy analysis**:
   - Identify optimal ESC voucher amount
   - Prioritize schools for ESC expansion
   - Estimate fiscal impact (voucher cost × new beneficiaries)